# 📑Tutorial-5: Sampled Signal Processing

**🌀signalproc Signal Processing Module**

Signal processing is crucial in side-channel analysis. Raw acquired physical signals (such as power consumption fluctuations and electromagnetic radiation) often contain a large amount of noise and redundant information. Signal processing strips away environmental interference and recovers the "physical fingerprint" of the chip's computation, greatly improving the signal-to-noise ratio and laying the foundation for subsequent attack analysis.

Common signal processing techniques include:
- **🎚️ Digital filtering:** Filter out environmental noise and retain the chip's true operating signal
- **📊 Horizontal analysis:** Use statistical methods (mean/variance) to observe the overall fluctuation patterns of the signal
- **🛠️ FFT frequency-domain conversion:** Time-frequency transformation to identify frequency-domain features
- **📍 Feature and peak detection:** Identify key features (such as encryption round operations)

In [1]:
import nuscar
import numpy as np

In [ ]:
reader = nuscar.ReaderETS("../datasets/sm4_stm32.ets")
ctn = nuscar.ContainerETS(reader, frame=range(8000)) # take only 8000 sample points
samples = (ctn[0:2].samples)
print(samples.shape)
fig = nuscar.plot(samples, resample=False, webgl=False)
fig

### Digital Filtering

In [ ]:
sample_lp = nuscar.signalproc.filter_lowpass(samples, fs = 1e9, cutoff=1e7) # fs sampling rate, cutoff filter cutoff frequency
sample_hp = nuscar.signalproc.filter_highpass(samples, fs = 1e9, cutoff=1e7)
sample_bp = nuscar.signalproc.filter_bandpass(samples, fs = 1e9, cutoff=(2e6, 1e7)) 
sample_bs = nuscar.signalproc.filter_bandstop(samples, fs = 1e9, cutoff=(2e6, 1e8))
sample_all = [sample_lp, sample_hp, sample_bp, sample_bs]
name_all = ['Lowpass', 'Highpass', 'Bandpass', 'Bandstop']

In [6]:
for s,n in zip(sample_all, name_all):
    fig = nuscar.plot(s, title=n, webgl=False)
    display(fig)

    'data': [{'hoverinfo': 'name+x+y',
              'line': {'color': '#636EFA'…

    'data': [{'hoverinfo': 'name+x+y',
              'line': {'color': '#636EFA'…

    'data': [{'hoverinfo': 'name+x+y',
              'line': {'color': '#636EFA'…

    'data': [{'hoverinfo': 'name+x+y',
              'line': {'color': '#636EFA'…

### Horizontal Analysis

In [ ]:
sample_mm = nuscar.signalproc.moving_mean(samples, 30)  # window_size = 30
sample_mv = nuscar.signalproc.moving_var(samples, 30)

In [9]:
nuscar.plot(sample_mm, title='moving mean', webgl=False)

    'data': [{'hoverinfo': 'name+x+y',
              'line': {'color': '#636EFA'…

In [10]:
nuscar.plot(sample_mv, title='moving var', webgl=False)

    'data': [{'hoverinfo': 'name+x+y',
              'line': {'color': '#636EFA'…

### FFT

In [ ]:
freq, magnitude = nuscar.signalproc.fft(data = samples, frequency = 1e9) # freq is the horizontal axis
nuscar.plot(x=freq, y=magnitude, webgl=False)

###  Feature and Peak Detection

Below we use pattern and peak detection to locate the cryptographic operations of interest

In [ ]:
nuscar.plot_pattern(samples[0], pattern_x=range(1100, 1700), webgl=False).show()  # samples[0, 1100:1700] is the cryptographic operation pattern of interest
pattern_ref = samples[0, 1100: 1720]

In [ ]:
pattern_match = nuscar.signalproc.pattern_detect(samples[1], pattern=pattern_ref, alg=nuscar.signalproc.PATTERN_ALG.DISTANCE) # use least squares for pattern detection
nuscar.plot(-1*pattern_match, webgl=False)

In [ ]:
peaks = nuscar.signalproc.peak_detect(-1*pattern_match, height=-200_000) # minimum height is -200K, filters matched positions
nuscar.plot_peak(pattern_match, peak_x=peaks, webgl=False) 

In [15]:
widget = nuscar.plot(samples[1], title='Sampled signal')
for p in peaks:
    widget.fig.add_vrect(x0=p, x1=p + 600, fillcolor='green', opacity=0.2, line_width=0)
widget

    'data': [{'hoverinfo': 'name+x+y',
              'line': {'color': '#636EFA'…